In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import re
import datetime
import random
from time import sleep
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'BT RMA'
print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running BT RMA Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
chromeOptions = webdriver.ChromeOptions()
prefs = {
    'plugins.always_open_pdf_externally': True,
    'download.prompt_for_download': False,
    'download.default_directory': tempfolder,
    'profile.default_content_setting_values.automatic_downloads': 1,
}
chromeOptions.add_experimental_option('prefs', prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
    'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 'InternalID_2_type': [],
    'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
    'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [],
    'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [],
    'CancellationDate': [], 'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [],
    'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [],
    'Name - Mother Company': [], 'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': []
}

processdate = now.strftime('%Y-%m-%d')

MAIN_URL = 'https://www.rma.org.bt/'

regdict = {
    regulatorName + ' 1': MAIN_URL,
    regulatorName + ' 2': MAIN_URL,
    regulatorName + ' 3': MAIN_URL,
}

Typology = {
    regulatorName + ' 1': 'List of Financial Institutions',
    regulatorName + ' 2': 'List of Microfinance Institutions',
    regulatorName + ' 3': 'List of Registered Private Money Lenders',
}

SECTION_KEYWORDS = {
    regulatorName + ' 1': ['financial institution', 'bank', 'non-bank', 'non bank'],
    regulatorName + ' 2': ['microfinance'],
    regulatorName + ' 3': ['private money lender', 'money lender'],
}

In [5]:
#------------------------------------------------ Begin_Function ----------------------------------------
def pad(d):
    maxlen = len(d['ListProcessDate'])
    for k in d:
        if len(d[k]) < maxlen:
            d[k] += [''] * (maxlen - len(d[k]))
    return d

def clean(text):
    if text is None:
        return ''
    text = str(text).replace('\xa0', ' ')
    text = re.sub(r'[\u200b-\u200f\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' \t\r\n-|;:,')

ADDR_SPLIT_RE = re.compile(r'\s*(?:;|\n|\r| / |\s\|\s|,?\s*(?:Branch(?:es)?|Address\s*2)\s*[:\-])\s*', re.IGNORECASE)

def first_address(value):
    """If the source exposes several addresses for one entity, keep only the first."""
    text = clean(value)
    if not text:
        return ''
    parts = [p.strip() for p in ADDR_SPLIT_RE.split(text) if p and p.strip()]
    return parts[0] if parts else text



def append_record(reg, list_name, name, bus_addr, per_addr, phone, email, website=''):
    name = clean(name)
    if not name:
        return False
    sqldict['Name'].append(name)
    sqldict['Address_1'].append(first_address(bus_addr))
    sqldict['Address_2'].append(first_address(per_addr) if per_addr  else '')
    sqldict['Phone'].append(phone if phone else '')
    sqldict['Email'].append(email if email else '')
    sqldict['Website'].append(clean(website))
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append('BT')
    sqldict['RegCode'].append('RMA')
    sqldict['ListCode'].append(reg.split(' ')[-1])
    sqldict['RegulationType'].append('Regulated')
    sqldict['ListName'].append(list_name)
    pad(sqldict)
    return True

In [6]:
#------------------------------------------------ Begin_Main: load homepage ----------------------------------------
driver.get(MAIN_URL)
wait = WebDriverWait(driver, 30)
wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
sleep(random.uniform(3, 5))

# Scroll to bottom so any lazy-loaded footer content renders
for _ in range(6):
    driver.execute_script('window.scrollBy(0, document.body.scrollHeight/6);')
    sleep(0.8)

soup = BeautifulSoup(driver.page_source, 'html.parser')
print(f"[INFO] Loaded {MAIN_URL} - {len(driver.page_source)} chars")

[INFO] Loaded https://www.rma.org.bt/ - 94846 chars


In [7]:
#------------------------------------------------ Begin_Main: parse FI + MFI sections ----------------------------------------
# Footer structure (div.footer-links):
#   <h4>Banks</h4>            <ul><li><a>Bank of Bhutan</a></li> ...</ul>
#   <h4>Non-Banks</h4>        <ul><li><a>RICBL</a></li> ...</ul>
#   <h4>Microfinance Institutions</h4>  <ul><li><a>...</a></li> ...</ul>
# List 1 = Banks + Non-Banks combined. List 2 = Microfinance Institutions.
# The footer exposes name (anchor text) and website (anchor href) per entity.
# No address info is provided in the footer, so Address_1 stays empty for these lists.

FOOTER_SECTIONS = {
    regulatorName + ' 1': ['banks', 'non-banks'],
    regulatorName + ' 2': ['microfinance institutions'],
}

def entries_under_h4(soup, headings):
    """Return list[dict(name, website)] from footer .footer-links blocks whose h4 matches."""
    wanted = {h.lower() for h in headings}
    entries = []
    for block in soup.select('div.footer-links'):
        h4 = block.find('h4')
        if h4 is None:
            continue
        if clean(h4.get_text(' ', strip=True)).lower() not in wanted:
            continue
        for a in block.select('ul li a[href]'):
            name = clean(a.get_text(' ', strip=True))
            website = urljoin(MAIN_URL, a['href']).strip()
            if name:
                entries.append({'name': name, 'website': website})
    return entries

for reg in [regulatorName + ' 1', regulatorName + ' 2']:
    list_name = Typology[reg]
    print(f"Working with {reg} - {list_name}")
    entries = entries_under_h4(soup, FOOTER_SECTIONS[reg])
    print(f"  [INFO] {len(entries)} names found in footer")
    added = 0
    seen = set()
    for e in entries:
        key = e['name'].lower()
        if key in seen:
            continue
        seen.add(key)
        if append_record(reg, list_name, e['name'], '', '', '', '', website=e['website']):
            added += 1
    print(f"  [INFO] {added} rows appended for {reg}")

Working with BT RMA 1 - List of Financial Institutions
  [INFO] 10 names found in footer
  [INFO] 10 rows appended for BT RMA 1
Working with BT RMA 2 - List of Microfinance Institutions
  [INFO] 7 names found in footer
  [INFO] 7 rows appended for BT RMA 2


In [8]:
#------------------------------------------------ Begin_Main: list 3 - PDF Registered Private Money Lenders ----------------------------------------
# Locate the PDF link by anchor text, download to tempfolder, then parse with pdfplumber.
reg = regulatorName + ' 3'
list_name = Typology[reg]
print(f"Working with {reg} - {list_name}")

pdf_url = None
for a in soup.find_all('a', href=True):
    txt = clean(a.get_text(' ', strip=True)).lower()
    href = a['href']
    if 'private money lender' in txt and href.lower().endswith('.pdf'):
        pdf_url = urljoin(MAIN_URL, href)
        break
    if 'private money lender' in txt and href.lower().endswith('.pdf') == False:
        # may be a landing page; remember as fallback
        pdf_url = pdf_url or urljoin(MAIN_URL, href)

if pdf_url is None:
    # known fallback per RMA legislation page
    pdf_url = 'https://www.rma.org.bt/media/Laws_By_Laws/Registered%20Private%20Money%20Lenders.pdf'

print(f"  [INFO] Downloading {pdf_url}")
pdf_path = os.path.join(tempfolder, 'private_money_lenders.pdf')
try:
    r = requests.get(pdf_url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=60,verify = False)
    r.raise_for_status()
    with open(pdf_path, 'wb') as f:
        f.write(r.content)
    print(f"  [INFO] Saved {len(r.content)} bytes")
except Exception as e:
    print(f"  [WARN] PDF download failed: {e}")
    pdf_path = None

Working with BT RMA 3 - List of Registered Private Money Lenders
  [INFO] Downloading https://www.rma.org.bt/media/Registered Private Money Lenders.pdf


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.rma.org.bt'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  [INFO] Saved 40822 bytes


In [9]:
# Parse the PDF: prefer tables with 'Business' / 'Name' / 'Address' headers; fallback to text-line parsing.
import pdfplumber

if pdf_path and os.path.exists(pdf_path):
    added = 0
    seen = set()
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables() or []
            for tbl in tables:
                if not tbl or len(tbl) < 2:
                    continue
                header = [clean(c).lower() for c in (tbl[0] or [])]
                # locate Business/Name column and Address column
                name_idx = next((i for i, h in enumerate(header) if 'business' in h), 0)
                bus_addr_idx = next((i for i, h in enumerate(header) if 'business address' in h ), None)
                per_addr_idx = next((i for i, h in enumerate(header) if 'permanent address' in h ), None)
                phone_idx = next((i for i, h in enumerate(header) if 'phone' in h), None)
                email_idx = next((i for i, h in enumerate(header) if 'email' in h), None)
                for row in tbl[1:]:
                    if not row:
                        continue
                    name = clean(row[name_idx]) if name_idx < len(row) else ''
                    bus_addr = clean(row[bus_addr_idx]) if (bus_addr_idx is not None and bus_addr_idx < len(row)) else ''
                    per_addr = clean(row[per_addr_idx]) if (per_addr_idx is not None and per_addr_idx < len(row)) else ''
                    phone = clean(row[phone_idx]) if (phone_idx is not None and phone_idx < len(row)) else ''
                    email = clean(row[email_idx]) if (email_idx is not None and email_idx < len(row)) else ''
                    if not name or name.lower() in {'business name', 'name', 'sl no', 's.no', 's. no', 'sl.no'}:
                        continue
                    if re.fullmatch(r'\d+\.?', name):
                        continue
                    key = name.lower()
                    if key in seen:
                        continue
                    seen.add(key)
                    if append_record(reg, list_name, name, bus_addr, per_addr, phone, email):
                        added += 1
    print(f"  [INFO] {added} rows appended for {reg}")
else:
    print(f"  [WARN] Skipping {reg} - PDF unavailable")

  [INFO] 5 rows appended for BT RMA 3


In [10]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
driver.quit()
sleep(2)
print(f"[INFO] Excel file '{filename}' saved successfully")

C:\Users\wuj1\AppData\Local\Temp\8\ipykernel_32108\3637743401.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] Excel file 'BT RMA SQL Ready 2026-05-19 11.00.35.xlsx' saved successfully


In [11]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------
print('=' * 70)
print('DATA INTEGRITY & CONSISTENCY VERIFICATION')
print('=' * 70)

expected_lists = {
    '1': 'List of Financial Institutions',
    '2': 'List of Microfinance Institutions',
    '3': 'List of Registered Private Money Lenders',
}

print(f"\nTotal rows: {len(df)}  |  Total columns: {len(df.columns)}")

print("\nDistribution by ListCode:")
if len(df):
    print(df.groupby('ListCode').agg(Count=('Name', 'count'), ListName=('ListName', 'first')))

print("\nRequired-field completeness:")
for col in ['Name', 'Address_1', 'RegCtry', 'RegCode', 'ListCode', 'ListName', 'ListProcessDate']:
    filled = (df[col].astype(str).str.strip() != '').sum() if len(df) else 0
    status = 'OK' if filled == len(df) and len(df) > 0 else ('EMPTY' if filled == 0 else 'PARTIAL')
    print(f"  {col:20s}: {filled}/{len(df)}  [{status}]")

print("\nRegCtry / RegCode values:")
print(f"  RegCtry: {sorted(set(df['RegCtry'])) if len(df) else '[]'} (expected ['BT'])")
print(f"  RegCode: {sorted(set(df['RegCode'])) if len(df) else '[]'} (expected ['RMA'])")

print("\nList coverage vs README:")
for code, expected_name in expected_lists.items():
    count = (df['ListCode'] == code).sum() if len(df) else 0
    status = 'COLLECTED' if count > 0 else 'MISSING'
    print(f"  List {code} - {expected_name:45s}  {count:4d} rows  [{status}]")

print("\nSample (first 5 rows):")
if len(df):
    print(df[['ListCode', 'Name', 'Address_1']].head(5).to_string(index=False))
print('=' * 70)

DATA INTEGRITY & CONSISTENCY VERIFICATION

Total rows: 22  |  Total columns: 44

Distribution by ListCode:
          Count                                  ListName
ListCode                                                 
1            10            List of Financial Institutions
2             7         List of Microfinance Institutions
3             5  List of Registered Private Money Lenders

Required-field completeness:
  Name                : 22/22  [OK]
  Address_1           : 5/22  [PARTIAL]
  RegCtry             : 22/22  [OK]
  RegCode             : 22/22  [OK]
  ListCode            : 22/22  [OK]
  ListName            : 22/22  [OK]
  ListProcessDate     : 22/22  [OK]

RegCtry / RegCode values:
  RegCtry: ['BT'] (expected ['BT'])
  RegCode: ['RMA'] (expected ['RMA'])

List coverage vs README:
  List 1 - List of Financial Institutions                   10 rows  [COLLECTED]
  List 2 - List of Microfinance Institutions                 7 rows  [COLLECTED]
  List 3 - List of Registere